# fqdn smoke test

Tests the `fqdn` package — RFC 1035/1123/2181-based validation of fully-qualified domain names.

In [ ]:
# Example 1: basic FQDN validation — valid vs invalid hostnames.
from fqdn import FQDN

valid = ['example.com', 'sub.example.org', 'deep.nested.domain.co.uk', 'valid-domain.com']
invalid = ['invalid..domain', '-bad-start.com', 'bad-end-.com', 'no_tld']

for d in valid:
    f = FQDN(d)
    print(f"{d:28s} -> is_valid={f.is_valid}")
    assert f.is_valid, f"{d} should be valid"

for d in invalid:
    f = FQDN(d)
    print(f"{d:28s} -> is_valid={f.is_valid}")
    assert not f.is_valid, f"{d} should be invalid"


In [ ]:
# Example 2: absolute (trailing-dot) vs relative forms.
from fqdn import FQDN

f = FQDN('example.com')
print(f"relative form: {f.relative}")
print(f"absolute form: {f.absolute}")
assert f.relative == 'example.com'
assert f.absolute == 'example.com.'

dotted = FQDN('example.com.')
print(f"'example.com.' is_valid_absolute: {dotted.is_valid_absolute}")
print(f"'example.com'  is_valid_absolute: {f.is_valid_absolute}")
print(f"'example.com'  is_valid_relative: {f.is_valid_relative}")
assert dotted.is_valid_absolute
assert not f.is_valid_absolute
assert f.is_valid_relative


In [ ]:
# Example 3: labels_count + the min_labels option (single-label hostnames).
from fqdn import FQDN

f = FQDN('deep.nested.domain.co.uk')
print(f"labels_count: {f.labels_count}")
assert f.labels_count == 5

# The default requires >= 2 labels, so a bare hostname is invalid...
assert not FQDN('localhost').is_valid
# ...but min_labels=1 accepts it.
assert FQDN('localhost', min_labels=1).is_valid
print("min_labels=1 accepts bare 'localhost'; the default (2) rejects it")


In [ ]:
# Example 4: allow_underscores for RFC-violating-but-common names (DNS service records).
from fqdn import FQDN

name = '_dmarc.example.com'
strict = FQDN(name)
relaxed = FQDN(name, allow_underscores=True)
print(f"{name} strict  -> {strict.is_valid}")
print(f"{name} relaxed -> {relaxed.is_valid}")
assert not strict.is_valid
assert relaxed.is_valid


In [ ]:
# Example 5: RFC length limits — 63-char label cap, 253-char total cap.
from fqdn import FQDN

label63 = 'a' * 63 + '.com'
label64 = 'a' * 64 + '.com'
print(f"63-char label valid: {FQDN(label63).is_valid}")
print(f"64-char label valid: {FQDN(label64).is_valid}")
assert FQDN(label63).is_valid
assert not FQDN(label64).is_valid

ok253 = 'a' * 63 + '.' + 'a' * 63 + '.' + 'a' * 63 + '.' + 'a' * 57 + '.com'
bad254 = 'a' * 63 + '.' + 'a' * 63 + '.' + 'a' * 63 + '.' + 'a' * 58 + '.com'
print(f"len={len(ok253)} valid: {FQDN(ok253).is_valid}")
print(f"len={len(bad254)} valid: {FQDN(bad254).is_valid}")
assert FQDN(ok253).is_valid
assert not FQDN(bad254).is_valid
